In [58]:
import pandas as pd
import numpy as np


In [59]:
df = pd.read_csv("dataset/Installment_shorter_sampled.csv")
df.head(10)

,Asset ID,Origination Date,Total Advance,Total EMI,Payment Date,Payment Amount
0,264,1/31/2024,"17,160","20,540",2/7/2024,860
1,264,1/31/2024,"17,160","20,540",2/14/2024,860
2,264,1/31/2024,"17,160","20,540",2/21/2024,860
3,264,1/31/2024,"17,160","20,540",2/28/2024,860
4,264,1/31/2024,"17,160","20,540",3/6/2024,860
5,264,1/31/2024,"17,160","20,540",3/13/2024,860
6,264,1/31/2024,"17,160","20,540",3/20/2024,860
7,264,1/31/2024,"17,160","20,540",3/27/2024,860
8,264,1/31/2024,"17,160","20,540",4/3/2024,860
9,264,1/31/2024,"17,160","20,540",4/10/2024,"9,000"


In [60]:
#data inspection
print("Raw shape:", df.shape)
print("Raw size:",df.size)
df.info()
print(df.columns.tolist())

Raw shape: (11894, 6)
Raw size: 71364
<class 'pandas.DataFrame'>
RangeIndex: 11894 entries, 0 to 11893
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Asset ID          11894 non-null  int64
 1   Origination Date  11894 non-null  str  
 2    Total Advance    11894 non-null  str  
 3    Total EMI        11894 non-null  str  
 4   Payment Date      11894 non-null  str  
 5    Payment Amount   11894 non-null  str  
dtypes: int64(1), str(5)
memory usage: 557.7 KB
['Asset ID', 'Origination Date', ' Total Advance ', ' Total EMI ', 'Payment Date', ' Payment Amount ']


In [61]:
#clean extra space from the column name
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())
#check
df["Total Advance"]

['Asset ID', 'Origination Date', 'Total Advance', 'Total EMI', 'Payment Date', 'Payment Amount']


0         17,160 
1         17,160 
2         17,160 
3         17,160 
4         17,160 
           ...   
11889      8,249 
11890      8,249 
11891      8,249 
11892     16,499 
11893     16,499 
Name: Total Advance, Length: 11894, dtype: str

In [62]:
#remove commas, spaces and convert to numeric
def clean_numeric(series):
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    cleaned = cleaned.replace("-", "0")    #"-" replace with 0 instead of dropping the whole column
    return pd.to_numeric(cleaned, errors="coerce")

df["Total Advance"] = clean_numeric(df["Total Advance"])
df["Total EMI"] = clean_numeric(df["Total EMI"])

#track how many installments were missed
missed_installments = (df["Payment Amount"].astype(str).str.strip() == "-").sum()
print(f"\nMissed/unpaid installment rows (recorded as '-'): {missed_installments}")

df["Payment Amount"] = clean_numeric(df["Payment Amount"])


Missed/unpaid installment rows (recorded as '-'): 4979


In [63]:
#check for numeric
print(df[["Total Advance", "Total EMI", "Payment Amount"]].dtypes)

Total Advance     int64
Total EMI         int64
Payment Amount    int64
dtype: object


In [64]:
#Convert date columns to datetime
df["Origination Date"] = pd.to_datetime(
    df["Origination Date"], format="%m/%d/%Y", errors="coerce"
)
df["Payment Date"] = pd.to_datetime(
    df["Payment Date"], format="%m/%d/%Y", errors="coerce"
)

#chevk
print(df[["Origination Date", "Payment Date"]].dtypes)

Origination Date    datetime64[us]
Payment Date        datetime64[us]
dtype: object


In [65]:
# remove exact duplicate rows
a = len(df)
df = df.drop_duplicates()
b = len(df)
print(f"Removed {a - b} exact duplicate rows")

Removed 638 exact duplicate rows


In [66]:
#Remove rows with invalid dates
a = len(df)
df = df.dropna(subset=["Origination Date", "Payment Date"])
b = len(df)
print(f"Removed {a - b} rows with invalid dates")

Removed 0 rows with invalid dates


In [67]:
#remove which payment date before origination date
before = len(df)
df = df[df["Payment Date"] >= df["Origination Date"]]
after = len(df)
print(f"Removed {before - after} rows where payment occurred before origination")

Removed 3 rows where payment occurred before origination


In [68]:
#check for outlier
df[["Total Advance", "Total EMI", "Payment Amount"]].describe()

,Total Advance,Total EMI,Payment Amount
count,11253.000000,11253.000000,11253.000000
mean,13072.591309,20282.096241,503.510175
std,3868.168687,6376.504288,721.823911
min,5760.000000,8100.000000,0.000000
25%,10050.000000,15381.000000,0.000000
50%,12920.000000,19910.000000,459.000000
75%,15840.000000,24300.000000,808.000000
max,29991.000000,48568.000000,16000.000000


Looks good and no extreme jump from 75% to max and aslo 25% for the min.

In [69]:
#Create cohort label (origination month) and elapsed month
df["Cohort"] = df["Origination Date"].dt.to_period("M").astype(str)

df["Elapsed Month"] = (
    (df["Payment Date"].dt.year - df["Origination Date"].dt.year) * 12
    + (df["Payment Date"].dt.month - df["Origination Date"].dt.month)
)

#save the copy
df_cleaned = df.copy()

In [70]:
#Also exclude cohorts that are clearly date-entry errors
cohort_loan_counts = df_cleaned.drop_duplicates(subset=["Asset ID"]).groupby("Cohort")["Asset ID"].nunique()
suspicious_cohorts = cohort_loan_counts[cohort_loan_counts <= 1].index.tolist()
if suspicious_cohorts:
    print(f"\nExcluding data-entry-error cohorts "
          f"outside main range): {suspicious_cohorts}")
    before = len(df_cleaned)
    df_cleaned = df_cleaned[~df_cleaned["Cohort"].isin(suspicious_cohorts)]
    print(f"Removed {before - len(df_cleaned)} rows from these cohorts")

#save this as csv
df_cleaned.to_csv("output/cleaned_installments.csv", index=False)
print("\nCleaned dataset saved")
print("Cleaned shape:", df_cleaned.shape)


Excluding data-entry-error cohorts outside main range): ['2025-09', '2025-10', '2026-01']
Removed 32 rows from these cohorts

Cleaned dataset saved
Cleaned shape: (11221, 8)


In [71]:
#total Advance per cohort
cohort_advance = (
    df_cleaned.drop_duplicates(subset=["Asset ID"])
    .groupby("Cohort")["Total Advance"]
    .sum()
    .rename("Cohort Total Advance")
)

print("\nTotal Advance per cohort:")
print(cohort_advance)


Total Advance per cohort:
Cohort
2024-01    102945
2024-02    136095
2024-03    795665
2024-04    368935
2024-05    224805
2024-06    447855
2024-07    286900
2024-08    325990
2024-09    408306
2024-10    461204
2024-11    634639
2024-12    267439
Name: Cohort Total Advance, dtype: int64
